In [2]:
import gpt as g
import numpy as np
import sys

In [29]:

# grid
L = [8, 8, 8, 8]
grid = g.grid(L, g.single)
grid_eo = g.grid(L, g.single, g.redblack)

# cold start
g.default.push_verbose("random", False)
rng = g.random("test", "vectorized_ranlux24_24_64")
U = [g.complex(grid) for i in range(4)]
for mu in range(len(U)):
    U[mu][:] = 1

# red/black mask
mask_rb = g.complex(grid_eo)
mask_rb[:] = 1

# full mask
mask = g.complex(grid)


# simple plaquette action
def staple(U, mu):
    st = g.lattice(U[0])
    st[:] = 0
    for nu in range(len(U)):
        if mu != nu:
            st += g.qcd.gauge.staple(U, mu, nu)
    return st


# 5 heatbath sweeps
beta = 1.1
g.default.push_verbose("u1_heat_bath", False)
markov = g.algorithms.markov.u1_heat_bath(rng)
for it in range(52):
    plaq = g.qcd.gauge.plaquette(U)
    g.message(f"U(1) heatbath {it} has P = {plaq}")
    for cb in [g.even, g.odd]:
        mask[:] = 0
        mask_rb.checkerboard(cb)
        g.set_checkerboard(mask, mask_rb)
        for mu in range(len(U)):
            st = g.eval(beta * staple(U, mu))
            markov(U[mu], st, mask)

#assert abs(plaq - 0.7133781214555105) < 1e-6

GPT :     566.559430 s : U(1) heatbath 0 has P = 1.0
GPT :     566.695183 s : U(1) heatbath 1 has P = 0.7728846867879232
GPT :     566.803155 s : U(1) heatbath 2 has P = 0.7296920617421468
GPT :     566.910439 s : U(1) heatbath 3 has P = 0.7245009740193685
GPT :     567.021194 s : U(1) heatbath 4 has P = 0.7133780320485433
GPT :     567.127758 s : U(1) heatbath 5 has P = 0.7142832279205322
GPT :     567.234814 s : U(1) heatbath 6 has P = 0.7096909681955973
GPT :     567.336143 s : U(1) heatbath 7 has P = 0.71962571144104
GPT :     567.441239 s : U(1) heatbath 8 has P = 0.7177567481994629
GPT :     567.547356 s : U(1) heatbath 9 has P = 0.717127243677775
GPT :     567.657680 s : U(1) heatbath 10 has P = 0.7192366123199463
GPT :     567.764663 s : U(1) heatbath 11 has P = 0.7272460460662842
GPT :     567.874367 s : U(1) heatbath 12 has P = 0.720438559850057
GPT :     567.977773 s : U(1) heatbath 13 has P = 0.7192606131235758
GPT :     568.081114 s : U(1) heatbath 14 has P = 0.72089799245

In [30]:
inv = g.algorithms.inverter
sympl = g.algorithms.integrator.symplectic

log = sympl.log()
pure_gauge = True

# conjugate momenta
mom = g.group.cartesian(U)

# Log
g.message(f"Lattice = {grid.fdimensions}")
g.message("Actions:")
# action for conj. momenta
a0 = g.qcd.scalar.action.mass_term()
g.message(f" - {a0.__name__}")

a1 = g.qcd.gauge.action.wilson(beta)
g.message(f" - {a1.__name__}")

def hamiltonian():
    return a0(mom) + a1(U) 


# molecular dynamics
sympl = g.algorithms.integrator.symplectic
    
iphmc = sympl.update_p(mom, lambda: a1.gradient(U, U))
iqhmc = sympl.update_q(U, lambda: a0.gradient(mom, mom))

# integrator
mdint_hmc = sympl.leap_frog(10, iphmc, iqhmc)
#g.message(f"Integration scheme:\n{mdint}")
    
# metropolis
metro = g.algorithms.markov.metropolis(rng)
    
# MD units
tau = 1.0
g.message(f"tau = {tau} MD units")

def hmc(tau, mom):
    rng.normal_element(mom)
    accrej = metro(U)
    h0 = hamiltonian()
    mdint_hmc(tau)
    h1 = hamiltonian()
    return [accrej(h1, h0), h1 - h0]

"""
# thermalization
for i in range(1, 20):
    h = []
    timer = g.timer("hmc")
    for _ in range(1):
        timer("trajectory")
        h += [hmc(tau, mom)]
    h = np.array(h)
    timer()
    g.message(f"{i*10} % of thermalization completed")
    g.message(timer)
    g.message(
        f"Plaquette = {g.qcd.gauge.plaquette(U)}, Acceptance = {np.mean(h[:,0]):.2f}, |dH| = {np.mean(np.abs(h[:,1])):.4e}"
    )


#start = time.time()

# production
history = []
plaq = []


for i in range(50):
    history += [hmc(tau, mom)]
    P = g.qcd.gauge.plaquette(U)
    plaq.append(P)
    g.message(f"Trajectory {i}, P={P}")

history = np.array(history)
g.message(f"Acceptance rate = {np.mean(history[:,0]):.2f}")
g.message(f"<|dH|> = {np.mean(np.abs(history[:,1])):.4e}")
"""

GPT :     572.211396 s : Lattice = [8, 8, 8, 8]
GPT :     572.211819 s : Actions:
GPT :     572.212057 s :  - mass_term(m^-1 = 1.0)
GPT :     572.212270 s :  - wilson(1.1)
GPT :     572.212608 s : tau = 1.0 MD units


'\n# thermalization\nfor i in range(1, 20):\n    h = []\n    timer = g.timer("hmc")\n    for _ in range(1):\n        timer("trajectory")\n        h += [hmc(tau, mom)]\n    h = np.array(h)\n    timer()\n    g.message(f"{i*10} % of thermalization completed")\n    g.message(timer)\n    g.message(\n        f"Plaquette = {g.qcd.gauge.plaquette(U)}, Acceptance = {np.mean(h[:,0]):.2f}, |dH| = {np.mean(np.abs(h[:,1])):.4e}"\n    )\n\n\n#start = time.time()\n\n# production\nhistory = []\nplaq = []\n\n\nfor i in range(50):\n    history += [hmc(tau, mom)]\n    P = g.qcd.gauge.plaquette(U)\n    plaq.append(P)\n    g.message(f"Trajectory {i}, P={P}")\n\nhistory = np.array(history)\ng.message(f"Acceptance rate = {np.mean(history[:,0]):.2f}")\ng.message(f"<|dH|> = {np.mean(np.abs(history[:,1])):.4e}")\n'

In [33]:
a = a1.gradient(U,U)

a[0][:][20]

array([0.+0.j], dtype=complex64)